# Phase 1: Colab Environment Setup

Set up the full SoARM + LIBERO + OpenVLA-OFT stack on a fresh Colab A100 runtime.

## Requirements

- [ ] ENV-01: All dependencies install in correct order without pip resolver conflicts
- [ ] ENV-02: EGL headless rendering configured — OffScreenRenderEnv produces non-black LIBERO frames
- [ ] ENV-03: OpenVLA-OFT loads on GPU (A100 bf16) and returns a valid 7-D action tensor

## Usage

**BLOCK A** (this block): Run cells 0-9 top to bottom, then restart the runtime.

**BLOCK B** (Plan 02): After restart, run verification cells for ENV-01 / ENV-02 / ENV-03.

> Note: Block A must complete fully before restarting. Do not run Block B cells before restart.

In [ ]:
import os

# ── USER CONFIGURATION ──────────────────────────────────────────────────────
# Set REPO_ROOT to the path where you cloned SoARM-Research.
# If using Google Drive: "/content/drive/MyDrive/SoARM-Research"
# If using git clone directly to Colab: "/content/SoARM-Research"
REPO_ROOT = "/content/drive/MyDrive/SoARM-Research"
# ────────────────────────────────────────────────────────────────────────────

# Derived path constants (do not edit these)
LIBERO_ROOT = f"{REPO_ROOT}/LIBERO/libero/libero"
LIBERO_PKG  = f"{REPO_ROOT}/LIBERO/libero"
OUT_DIR     = f"{REPO_ROOT}/LIBERO/notebooks/outputs"
BDDL_FILE   = (
    f"{LIBERO_ROOT}/bddl_files/libero_spatial/"
    "pick_up_the_black_bowl_from_table_center_and_place_it_on_the_plate.bddl"
)

# Create outputs directory so Block B render check can save there
os.makedirs(OUT_DIR, exist_ok=True)

print(f"REPO_ROOT   = {REPO_ROOT}")
print(f"LIBERO_ROOT = {LIBERO_ROOT}")
print(f"LIBERO_PKG  = {LIBERO_PKG}")
print(f"OUT_DIR     = {OUT_DIR}")
print(f"BDDL_FILE   = {BDDL_FILE}")
print(f"Saved → {OUT_DIR}  (outputs directory ready)")

In [ ]:
# GPU assertion — D-06
# Check GPU availability and warn loudly if not A100.
# OpenVLA-OFT in bf16 requires ~16 GB VRAM; A100 (40 GB) is the target.
import torch

assert torch.cuda.is_available(), (
    "No GPU available. Go to Runtime > Change runtime type > Hardware accelerator > GPU."
)

gpu_name = torch.cuda.get_device_name(0)
vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9

print(f"GPU:  {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

if "A100" not in gpu_name:
    print()
    print("WARNING: Expected A100, got", gpu_name)
    print("WARNING: OpenVLA-OFT in bf16 requires ~16 GB+ VRAM.")
    print("WARNING: ENV-03 will OOM on T4 (15 GB). Restart with an A100 runtime.")
    print("WARNING: You may still proceed for install-only testing on T4.")
else:
    print("A100 confirmed. Proceeding.")

---

## BLOCK A: Install

Run all cells in this block **top to bottom**, then restart the runtime.

**Ordering is critical** — do not reorder or skip cells:

1. EGL system packages (apt) must install before pip mujoco
2. PyTorch must install before flash-attn (flash-attn compiles against torch CUDA headers)
3. The custom transformers fork must install last (prevents pip downgrade to PyPI version)

---

In [ ]:
# Step 1 of 6 — EGL system packages
# Must run BEFORE pip mujoco install.
# These C libraries must exist when the mujoco Python extension builds.
!apt-get install -y -q \
    libglfw3 \
    libglew-dev \
    libosmesa6-dev \
    libgles2 \
    libglvnd0 \
    libegl-dev \
    libegl1 \
    libgl1-mesa-glx

In [ ]:
# Step 2 of 6 — PyTorch 2.2.0 (cu121)
# Must run BEFORE flash-attn: flash-attn compiles CUDA kernels against installed torch headers.
!pip install torch==2.2.0 torchvision==0.17.0 torchaudio==2.2.0 \
    --index-url https://download.pytorch.org/whl/cu121 -q

In [ ]:
# Step 3 of 6 — MuJoCo + simulation stack
# Exact versions from LIBERO/requirements.txt.
# robosuite must be 1.4.0 — 1.5.x removed SingleArmEnv which LIBERO depends on.
!pip install mujoco==2.3.7 gym==0.25.2 -q
!pip install \
    robosuite==1.4.0 \
    bddl==1.0.1 \
    easydict==1.9 \
    cloudpickle==2.1.0 \
    einops==0.4.1 \
    numpy==1.22.4 \
    opencv-python==4.6.0.66 \
    "imageio[ffmpeg]" -q

In [ ]:
# Step 4 of 6 — LIBERO editable install
# editable install — LIBERO package changes are live without reinstall
# LIBERO_PKG is defined in cell 1 (REPO_ROOT config cell).
import subprocess, sys

result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-e", LIBERO_PKG, "-q"],
    capture_output=True,
    text=True,
)
print(result.stdout[-500:] if result.stdout else "(no stdout)")
if result.returncode != 0:
    print("STDERR:", result.stderr[-500:])

In [ ]:
# Step 5 of 6 — OpenVLA-OFT supporting packages + custom transformers fork
# The git fork (moojink/transformers-openvla-oft) MUST be installed LAST in this cell.
# Do NOT install transformers from PyPI — the fork replaces it entirely.
# The fork adds bidirectional attention for parallel decoding; PyPI version lacks this.
!pip install \
    timm==0.9.10 \
    tokenizers==0.19.1 \
    sentencepiece==0.1.99 \
    peft==0.11.1 \
    accelerate \
    huggingface_hub -q

# Install transformers fork LAST — pip resolver cannot downgrade to PyPI version this way.
# Commit SHA comment for reproducibility: installs from main branch of the fork repo.
# To pin a specific commit: git+https://github.com/moojink/transformers-openvla-oft.git@<SHA>
!pip install git+https://github.com/moojink/transformers-openvla-oft.git -q

In [ ]:
# Step 6 of 6 — flash-attn (must be last install)
# This cell takes ~10 min to compile CUDA kernels. Do not interrupt.
# Must run AFTER torch is installed: flash-attn compiles against torch CUDA headers.
!pip install packaging ninja -q
!pip install "flash-attn==2.5.5" --no-build-isolation -q

---

## *** STOP — Restart runtime now ***

Go to: **Runtime > Restart session** (or press Ctrl+M .), then continue from BLOCK B below.

Do **not** run any cells below this point until after the runtime has restarted.

After restart, open notebook **02-colab-verify.ipynb** (Plan 02) for Block B verification.

---